# Gem vs control median signal delta mapped to the center of each 9-mer

This notebook generates a forward-strand-only figure similar to `sample_image.png` for the ligation product reference.

Main choices used here:
- DNAscent align rows are treated as full 9-mer signal contexts.
- Each residual is assigned to the **middle base of its 9-mer window**: for example, the final window `607-615` is plotted at center position `611`.
- The gemcitabine site is assumed to be the **second base from the end**, so position **614**. Position 614 is not drawn as a site line because the x-axis now shows 9-mer centers, not the modified base itself.
- Instead, windows containing position 614 are highlighted. For this reference, those terminal windows are expected to be `606-614` and `607-615`, centered at positions `610` and `611`.
- Per-read residuals are summarized the same way as the earlier scripts: within each read and position, multiple events are averaged first; then the cross-read **median** and **IQR** are plotted.


In [ ]:
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('default')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

DATA_DIR = Path.cwd()
if not (DATA_DIR / 'ligation_product.fasta').exists():
    DATA_DIR = Path('/scratch/esevinc22/DNAscent/DNAscent/Eylul_gem_data')

REF_FASTA = DATA_DIR / 'ligation_product.fasta'
GEM_ALIGN = DATA_DIR / 'gem' / 'dnascent_align' / 'gem.l600.test.align'
CONTROL_ALIGN = DATA_DIR / 'control' / 'dnascent_align' / 'control.l600.test.align'
OUTPUT_DIR = DATA_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

PLOT_START = 600
KMER_SIZE = 9
CENTER_OFFSET = KMER_SIZE // 2
FORWARD_TAG = 'fwd'

REF_FASTA, GEM_ALIGN, CONTROL_ALIGN, OUTPUT_DIR

In [ ]:
def read_fasta_sequence(path: Path) -> tuple[str, str]:
    header = None
    parts = []
    with path.open() as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                header = line[1:]
            else:
                parts.append(line)
    if header is None:
        raise ValueError(f'No FASTA header found in {path}')
    return header, ''.join(parts)


def build_center_position_mapping(sequence: str, start_center_pos: int, end_center_pos: int, gem_site_pos: int, kmer_size: int = 9) -> pd.DataFrame:
    # DNAscent align `coord` behaves like the 0-based center coordinate of the 9-mer.
    # This mapping therefore assigns each full 9-mer signal to its middle base.
    seq_len = len(sequence)
    center_offset = kmer_size // 2
    rows = []
    for center_pos1 in range(start_center_pos, end_center_pos + 1):
        start1 = center_pos1 - center_offset
        end1 = center_pos1 + center_offset
        if start1 < 1 or end1 > seq_len:
            raise ValueError(f'Center position {center_pos1} does not have a full {kmer_size}-mer window')

        expected_9mer = sequence[start1 - 1:end1]
        rows.append(
            {
                'genomic_pos_1based': center_pos1,
                'align_coord_0based': center_pos1 - 1,
                'align_coord_1based_center': center_pos1,
                'kmer_start_1based': start1,
                'kmer_end_1based': end1,
                'expected_9mer': expected_9mer,
                'base_at_center': sequence[center_pos1 - 1],
                'contains_gem_site': start1 <= gem_site_pos <= end1,
                'window_label': f'{start1}-{end1}',
            }
        )
    return pd.DataFrame(rows)


def parse_align_header(header_line: str) -> dict:
    parts = header_line[1:].strip().split()
    return {
        'read_id': parts[0] if len(parts) > 0 else None,
        'reference_name': parts[1] if len(parts) > 1 else None,
        'start': int(parts[2]) if len(parts) > 2 else None,
        'end': int(parts[3]) if len(parts) > 3 else None,
        'strand': parts[4] if len(parts) > 4 else None,
    }


def collect_forward_position_residuals(align_path: Path, mapping_df: pd.DataFrame, strand_tag: str = 'fwd'):
    mapping_rows = mapping_df.to_dict(orient='records')
    coord_to_positions = defaultdict(list)
    expected_kmer_by_coord = {}
    for row in mapping_rows:
        coord_to_positions[row['align_coord_0based']].append(row['genomic_pos_1based'])
        expected_kmer_by_coord[row['align_coord_0based']] = row['expected_9mer']

    per_position_read_means = defaultdict(list)
    current_header = None
    current_is_forward = False
    per_read_values = defaultdict(list)
    total_reads = 0
    forward_reads = 0
    kmer_mismatch_rows = 0

    def flush_read():
        if not current_is_forward:
            return
        for pos1, values in per_read_values.items():
            if values:
                per_position_read_means[pos1].append(float(np.mean(values)))

    with align_path.open() as handle:
        for line in handle:
            if not line.strip():
                continue
            if line.startswith('>'):
                if current_header is not None:
                    flush_read()
                current_header = parse_align_header(line)
                total_reads += 1
                current_is_forward = current_header['strand'] == strand_tag
                if current_is_forward:
                    forward_reads += 1
                per_read_values = defaultdict(list)
                continue

            if not current_is_forward:
                continue

            fields = line.rstrip('\n').split('\t')
            if len(fields) < 5:
                continue

            coord0 = int(fields[0])
            if coord0 not in coord_to_positions:
                continue

            kmer = fields[1]
            expected_kmer = expected_kmer_by_coord[coord0]
            if kmer != expected_kmer:
                kmer_mismatch_rows += 1
                continue

            observed = float(fields[2])
            expected = float(fields[4])
            residual = observed - expected
            for pos1 in coord_to_positions[coord0]:
                per_read_values[pos1].append(residual)

    if current_header is not None:
        flush_read()

    return per_position_read_means, {
        'total_reads': total_reads,
        'forward_reads': forward_reads,
        'kmer_mismatch_rows': kmer_mismatch_rows,
    }


def summarize_position_residuals(sample_name: str, residuals_by_position: dict, mapping_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for row in mapping_df.to_dict(orient='records'):
        pos1 = row['genomic_pos_1based']
        values = np.asarray(residuals_by_position.get(pos1, []), dtype=float)
        rows.append(
            {
                'sample': sample_name,
                'genomic_pos_1based': pos1,
                'align_coord_0based': row['align_coord_0based'],
                'kmer_start_1based': row['kmer_start_1based'],
                'kmer_end_1based': row['kmer_end_1based'],
                'expected_9mer': row['expected_9mer'],
                'base_at_center': row['base_at_center'],
                'contains_gem_site': row['contains_gem_site'],
                'window_label': row['window_label'],
                'n_reads': int(values.size),
                'median_delta': float(np.median(values)) if values.size else np.nan,
                'q1_delta': float(np.quantile(values, 0.25)) if values.size else np.nan,
                'q3_delta': float(np.quantile(values, 0.75)) if values.size else np.nan,
                'mean_delta': float(values.mean()) if values.size else np.nan,
                'std_delta': float(values.std(ddof=1)) if values.size > 1 else np.nan,
            }
        )
    return pd.DataFrame(rows)


In [ ]:
ref_name, reference_sequence = read_fasta_sequence(REF_FASTA)
reference_length = len(reference_sequence)
gem_site_pos = reference_length - 1
plot_end = reference_length - CENTER_OFFSET  # Final full 9-mer is 607-615, centered at 611.

mapping_df = build_center_position_mapping(
    reference_sequence,
    start_center_pos=PLOT_START,
    end_center_pos=plot_end,
    gem_site_pos=gem_site_pos,
    kmer_size=KMER_SIZE,
)

print(f'Reference: {ref_name}')
print(f'Reference length: {reference_length} bp')
print(f'Gemcitabine site (second from end): position {gem_site_pos}')
print(f'Plotting 9-mer center positions: {PLOT_START}-{plot_end}')
print('Gem-containing windows:', ', '.join(mapping_df.loc[mapping_df['contains_gem_site'], 'window_label']))
mapping_df

In [ ]:
gem_residuals, gem_info = collect_forward_position_residuals(GEM_ALIGN, mapping_df, strand_tag=FORWARD_TAG)
control_residuals, control_info = collect_forward_position_residuals(CONTROL_ALIGN, mapping_df, strand_tag=FORWARD_TAG)

print('Gem info:', gem_info)
print('Control info:', control_info)

assert gem_info['kmer_mismatch_rows'] == 0, f"Gem align/reference k-mer mismatches: {gem_info['kmer_mismatch_rows']}"
assert control_info['kmer_mismatch_rows'] == 0, f"Control align/reference k-mer mismatches: {control_info['kmer_mismatch_rows']}"

gem_summary = summarize_position_residuals('gem', gem_residuals, mapping_df)
control_summary = summarize_position_residuals('control', control_residuals, mapping_df)

summary_df = (
    gem_summary.merge(
        control_summary,
        on=['genomic_pos_1based', 'align_coord_0based', 'kmer_start_1based', 'kmer_end_1based', 'expected_9mer', 'base_at_center', 'contains_gem_site', 'window_label'],
        suffixes=('_gem', '_control'),
    )
    .sort_values('genomic_pos_1based')
    .reset_index(drop=True)
)

summary_path = OUTPUT_DIR / 'gem_vs_control.forward.600_to_end.median_iqr.tsv'
summary_df.to_csv(summary_path, sep='\t', index=False)
summary_df

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.6))

x = summary_df['genomic_pos_1based'].to_numpy()
gem_window_rows = summary_df[summary_df['contains_gem_site']]
for idx, row in gem_window_rows.iterrows():
    center_pos = row['genomic_pos_1based']
    ax.axvspan(
        center_pos - 0.45,
        center_pos + 0.45,
        color='#f0c419',
        alpha=0.22,
        linewidth=0,
        label='9-mer contains gem site' if idx == gem_window_rows.index[0] else None,
    )

ax.plot(
    x,
    summary_df['median_delta_control'],
    color='#377eb8',
    marker='o',
    linewidth=1.6,
    markersize=5.5,
    label='Control',
)
ax.fill_between(
    x,
    summary_df['q1_delta_control'],
    summary_df['q3_delta_control'],
    color='#377eb8',
    alpha=0.18,
    label='Control IQR',
)

ax.plot(
    x,
    summary_df['median_delta_gem'],
    color='#d62728',
    marker='s',
    linewidth=1.6,
    markersize=5.2,
    label='Gemcitabine',
)
ax.fill_between(
    x,
    summary_df['q1_delta_gem'],
    summary_df['q3_delta_gem'],
    color='#d62728',
    alpha=0.18,
    label='Gemcitabine IQR',
)

ax.axhline(0, color='gray', linestyle='--', linewidth=0.8, label='No shift')

y_min = np.nanmin([summary_df['q1_delta_control'].min(), summary_df['q1_delta_gem'].min()])
y_max = np.nanmax([summary_df['q3_delta_control'].max(), summary_df['q3_delta_gem'].max()])
label_y = y_max + 0.04 * (y_max - y_min)
for _, row in gem_window_rows.iterrows():
    ax.text(
        row['genomic_pos_1based'],
        label_y,
        row['window_label'],
        ha='center',
        va='bottom',
        fontsize=8,
        color='#5a4300',
    )

ax.set_xlim(PLOT_START - 0.5, plot_end + 0.5)
ax.set_xticks(np.arange(PLOT_START, plot_end + 1, 1))
ax.set_ylim(y_min - 0.08 * (y_max - y_min), y_max + 0.16 * (y_max - y_min))
ax.set_xlabel('9-mer center position (1-based)')
ax.set_ylabel('Median delta (observed - expected)')
ax.set_title('Median signal delta mapped to 9-mer centers — gem vs control (forward strand only)')
ax.legend(loc='best', frameon=True)
ax.grid(False)
fig.tight_layout()

plot_path = OUTPUT_DIR / 'gem_vs_control.forward.600_to_end.median_iqr.png'
fig.savefig(plot_path, dpi=200, bbox_inches='tight')
plot_path

In [ ]:
summary_df[['genomic_pos_1based', 'base_at_center', 'kmer_start_1based', 'kmer_end_1based', 'contains_gem_site', 'expected_9mer', 'n_reads_gem', 'median_delta_gem', 'q1_delta_gem', 'q3_delta_gem', 'n_reads_control', 'median_delta_control', 'q1_delta_control', 'q3_delta_control']]